# Compute Reviewer Tables

**Paper:** Distribution-Free Recalibration of Tail Quantile Forecasts under Temporal Dependence

Computes three tables added in response to reviewer comments:
1. **Multi-Quantile** — Conformal correction at alpha in {0.01, 0.05, 0.10}
2. **Panel Pooled** — Cluster-robust pooled backtest across 9 assets
3. **Pinball Loss** — Quantile Score for all models + GARCH benchmarks

**Repository:** [QuantLet/Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle)

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

BASE_DIR = "/Users/danielpele/Documents/CFP LLM VaR"
ASSETS_DIR = os.path.join(BASE_DIR, "Assets data")
MODELS = {
    "GPT-3.5": {"dir": os.path.join(BASE_DIR, "gpt3.5_simulations")},
    "GPT-4":   {"dir": os.path.join(BASE_DIR, "gpt4_simulations")},
    "GPT-4o":  {"dir": os.path.join(BASE_DIR, "gpt_4o_simulations")},
}
ASSETS = ["CRIX", "SP500", "SPGTCLTR", "stoxx", "cact", "gdaxi", "cbu", "ftse", "djci"]
F_CAL = 0.70
W = 30

## Data Loading

In [ ]:
def find_csv(model_dir, asset, w):
    for f in os.listdir(model_dir):
        if not f.endswith(".csv"): continue
        if f"_w={w}.csv" in f and (f"_{asset}_" in f or f"_{asset.lower()}_" in f or f"_{asset.upper()}_" in f):
            return os.path.join(model_dir, f)
    for f in os.listdir(model_dir):
        if not f.endswith(".csv"): continue
        if f"w={w}" in f and asset.lower() in f.lower():
            return os.path.join(model_dir, f)
    return None

def load_simulation(model_name, asset, w):
    fpath = find_csv(MODELS[model_name]["dir"], asset, w)
    if fpath is None: return None, None
    df = pd.read_csv(fpath, parse_dates=["Date"], encoding="latin-1")
    return df["Date"].values, df.iloc[:, 1:].values

def load_realized_returns(asset):
    fmap = {"CRIX":"CRIX.xlsx","SP500":"SP500.xlsx","SPGTCLTR":"SPGTCLTR.xlsx",
            "stoxx":"stoxx.xlsx","cact":"cact.xlsx","gdaxi":"gdaxi.xlsx",
            "cbu":"cbu.xlsx","ftse":"ftse.xlsx","djci":"djci.xlsx"}
    df = pd.read_excel(os.path.join(ASSETS_DIR, fmap[asset]))
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))
    return df[["Date", "log_return"]].dropna().reset_index(drop=True)

def align_data(sim_dates, samples, ret_df):
    common = np.intersect1d(pd.to_datetime(sim_dates), ret_df["Date"].values)
    return samples[np.isin(pd.to_datetime(sim_dates), common)], ret_df.loc[np.isin(ret_df["Date"].values, common), "log_return"].values

# Load all data
data = {}
for model in MODELS:
    for asset in ASSETS:
        sd, sa = load_simulation(model, asset, W)
        if sd is None: print(f"Missing {model} {asset}"); continue
        s, r = align_data(sd, sa, load_realized_returns(asset))
        data[(model, asset)] = (s, r)
print(f"Loaded {len(data)} combinations.")

## Core Functions

In [ ]:
def conformal_calibrate(samples, realized, alpha, f_cal=F_CAL):
    T = len(samples)
    n_cal = int(T * f_cal)
    q_lo = np.array([np.quantile(samples[t][~np.isnan(samples[t])], alpha) for t in range(T)])
    s_V = q_lo[:n_cal] - realized[:n_cal]
    s_valid = s_V[~np.isnan(s_V)]
    qV = np.quantile(s_valid, min(np.ceil((len(s_valid)+1)*(1-alpha))/len(s_valid), 1.0)) if len(s_valid)>0 else 0.0
    tr, tq = realized[n_cal:], q_lo[n_cal:]
    return dict(q_hat_V=qV, n_test=len(tr), n_cal=n_cal, raw_pi=np.mean(tr<tq),
                corr_pi=np.mean(tr<(tq-qV)), raw_violations=tr<tq, corr_violations=tr<(tq-qV),
                test_q_lo=tq, corrected_q=tq-qV, test_realized=tr)

def kupiec_pvalue(pi_hat, n, alpha):
    x = int(round(pi_hat * n))
    if x==0: return 1.0
    if x==n: return 0.0
    lr = -2*(np.log((1-alpha)**(n-x)*alpha**x) - np.log((1-x/n)**(n-x)*(x/n)**x))
    return 1 - stats.chi2.cdf(lr, 1)

def kupiec_power(n, alpha_true, alpha_test, n_sim=10000):
    rej = 0
    for _ in range(n_sim):
        x = np.random.binomial(n, alpha_true)
        p = kupiec_pvalue(x/n, n, alpha_test) if 0<x<n else (1.0 if x==0 else 0.0)
        if p < 0.05: rej += 1
    return rej / n_sim

def quantile_score(q_hat, realized, alpha):
    return np.mean((alpha - (realized < q_hat).astype(float)) * (q_hat - realized))

## Table 1: Multi-Quantile (GPT-4, w=30)

In [ ]:
mq = []
for alpha in [0.10, 0.05, 0.01]:
    rp, cp, qvs, nts, kps = [], [], [], [], []
    for a in ASSETS:
        if ("GPT-4",a) not in data: continue
        r = conformal_calibrate(*data[("GPT-4",a)], alpha)
        rp.append(r["raw_pi"]); cp.append(r["corr_pi"]); qvs.append(r["q_hat_V"])
        nts.append(r["n_test"]); kps.append(kupiec_pvalue(r["corr_pi"], r["n_test"], alpha))
    mn = np.mean(nts)
    pw = kupiec_power(int(mn), 2*alpha, alpha, 10000)
    mq.append(dict(alpha=alpha, exp_viol=alpha*mn, raw_pi=np.mean(rp), qV=np.mean(qvs),
                   corr_pi=np.mean(cp), kup_p=np.mean(kps), power=pw))

pd.DataFrame(mq).round(3)

## Table 2: Panel Pooled (w=30, alpha=0.01)

In [ ]:
panel = []
for model in ["GPT-3.5", "GPT-4", "GPT-4o"]:
    rv, cv, ar, np_ = [], [], [], 0
    for a in ASSETS:
        if (model,a) not in data: continue
        r = conformal_calibrate(*data[(model,a)], 0.01)
        np_ += r["n_test"]; rv.append(sum(r["raw_violations"])); cv.append(sum(r["corr_violations"]))
        ar.append(r["corr_pi"])
    rpi, cpi = sum(rv)/np_, sum(cv)/np_
    vj = np.array(ar)
    se = np.sqrt(np.var(vj-0.01, ddof=1)/len(vj))
    z = (cpi - 0.01)/se if se>0 else 0
    p = 1 - stats.norm.cdf(z)  # one-sided H1: pi > alpha
    panel.append(dict(model=model, N=np_, raw_pi=rpi, corr_pi=cpi, p_pooled=p))

pd.DataFrame(panel).round(3)

## Table 3: Quantile Score / Pinball Loss (w=30, alpha=0.01)

In [ ]:
from arch import arch_model

pin = []
for model in ["GPT-3.5", "GPT-4", "GPT-4o"]:
    qr, qc = [], []
    for a in ASSETS:
        if (model,a) not in data: continue
        r = conformal_calibrate(*data[(model,a)], 0.01)
        qr.append(quantile_score(r["test_q_lo"], r["test_realized"], 0.01))
        qc.append(quantile_score(r["corrected_q"], r["test_realized"], 0.01))
    pin.append(dict(model=model, QS_raw=abs(np.mean(qr))*1e4, QS_corr=abs(np.mean(qc))*1e4))

# GARCH benchmarks
gqs, tqs = [], []
for a in ASSETS:
    if ("GPT-4",a) not in data: continue
    rdf = load_realized_returns(a)
    ar = rdf["log_return"].values
    T = len(data[("GPT-4",a)][1]); nc = int(T*F_CAL); nt = T-nc
    ts = len(ar)-nt; tr = ar[ts:]
    gq, tq = [], []
    for i in range(ts, len(ar)):
        s = max(0,i-250); train = ar[s:i]*100
        if len(train)<50: gq.append(np.nan); tq.append(np.nan); continue
        try:
            m = arch_model(train, vol="Garch", p=1, q=1, dist="normal", mean="Zero")
            f = m.fit(disp="off", show_warning=False)
            sig = np.sqrt(f.forecast(horizon=1).variance.values[-1,0])
            gq.append(stats.norm.ppf(0.01)*sig/100)
        except: gq.append(np.nan)
        try:
            m = arch_model(train, vol="Garch", p=1, q=1, dist="studentst", mean="Zero")
            f = m.fit(disp="off", show_warning=False)
            sig = np.sqrt(f.forecast(horizon=1).variance.values[-1,0])
            nu = f.params.get("nu",5)
            tq.append(stats.t.ppf(0.01,nu)*sig/100)
        except: tq.append(np.nan)
    gq, tq = np.array(gq), np.array(tq)
    v = ~np.isnan(gq)
    if v.sum()>0: gqs.append(quantile_score(gq[v], tr[v], 0.01))
    v = ~np.isnan(tq)
    if v.sum()>0: tqs.append(quantile_score(tq[v], tr[v], 0.01))

gm, tm = abs(np.mean(gqs))*1e4, abs(np.mean(tqs))*1e4
for r in pin: r["QS_GARCH_N"] = gm; r["QS_GAS_t"] = tm

pd.DataFrame(pin).round(2)